# Hybrid NER + Classification Pipeline for OJT Journal Task Tagging
### Deterministic Dictionary + Contextual Transformer NER with TRTR vs. TRSTR Ablation

This notebook provides an end-to-end, reproducible workflow for the OJT Task Extraction system:
1. **Environment Setup & GPU Acceleration**: PyTorch 2.6.0+cu124, spaCy 3.8.16, transformers.
2. **Real Annotated Data Ingestion**: Cleaned student journal entries (`data/data.jsonl`).
3. **Seed Terms Dictionary**: Seed lookup (`data/terms.csv`) normalized to `IT_TERM` / `CLERICAL_TERM`.
4. **Leak-Free Real Dataset Partitioning**: Sentence-level connected-component splitting (`train.spacy`, `dev.spacy`, `test.spacy`).
5. **Paraphrase-Based Synthetic Augmentation**: T5-driven syntactic diversification anchored to the training partition (`train_trstr.spacy`).
6. **7-Check Data Leakage Audit**: Strict validation across splits, unseen benchmark terms, and synthetic near-duplicates (similarity $\ge 0.70$).
7. **Transformer Fine-Tuning**: Patience-based early stopping on GPU for **TRTR** (Real Only) and **TRSTR** (Real + Synthetic).
8. **Side-by-Side Comparative Evaluation**: Testing on real held-out test data and the 65-term unseen benchmark.
9. **Thesis Synthesis & Methodological Findings**: Quantifying generalization lift and documentation.


---
## Phase 1: Environment Setup & GPU Initialization

Import core modules and initialize GPU acceleration on the NVIDIA RTX 3060 Laptop GPU.


In [1]:
import os
import sys
import json
import pandas as pd
import spacy

# Add project root to sys.path
sys.path.insert(0, os.path.abspath("."))

import scripts
from scripts.annotation import (
    load_terms_dictionary,
    find_term_spans,
    load_jsonl,
    report_dataset_diagnostics,
    prepare_real_data_pipeline,
)
from scripts.generate_synthetic_augmentation import run_synthetic_pipeline
from scripts.training import train_ner_trf
from scripts.pipeline import HybridJournalPipeline
from scripts.eval import (
    evaluate_unseen_generalization,
    evaluate_test_docbin,
    evaluate_real_holdout,
    load_unseen_benchmark,
)
from scripts.check_data_leakage import run_all_leakage_checks

# Initialize GPU acceleration
gpu_ready = scripts.init_gpu()
print(f"System Status: GPU Acceleration Active = {gpu_ready}")


/home/caineirb/Documents/PauPau/spaCy-training/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-09-23 12:57:30,117] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)


System Status: GPU Acceleration Active = True


---
## Phase 2: Load & Inspect Real Annotated Data

Load the authentic, manually annotated OJT student journal dataset (`data/data.jsonl`) and audit its composition diagnostics (negative ratio target: 25–35%, class balance ratio $\le 1.5:1$).


In [2]:
data_path = "data/data.jsonl"
records = load_jsonl(data_path)
print(f"Loaded {len(records)} records from {data_path}\n")

# Report dataset diagnostics (negative ratio, class balance)
diagnostics = report_dataset_diagnostics(records, dataset_label="Real Annotated Dataset")

# Display sample records with highlighted entity spans
print("\nSample Records:")
for r in records[:5]:
    text = r["text"]
    ents = [(text[e["start"]:e["end"]], e["label"]) for e in r.get("entities", [])]
    print(f"  Text    : {text}")
    print(f"  Entities: {ents if ents else '(Negative — 0 entities)'}\n")


[2026-09-23 12:57:30,127] INFO: --- Real Annotated Dataset Diagnostics ---
[2026-09-23 12:57:30,128] INFO:   Total records : 1044
[2026-09-23 12:57:30,128] INFO:   Positives     : 725
[2026-09-23 12:57:30,129] INFO:   Negatives     : 319 (30.6%)
[2026-09-23 12:57:30,129] INFO:   Entity labels : {'CLERICAL_TERM': 528, 'IT_TERM': 514}
[2026-09-23 12:57:30,129] INFO:   Negative ratio is within target range (25-35%).
[2026-09-23 12:57:30,130] INFO:   Class balance acceptable: IT_TERM=514, CLERICAL_TERM=528 (ratio 1.03:1)


Loaded 1044 records from data/data.jsonl


Sample Records:
  Text    : Encoded and organized vendor evaluation data using Microsoft Excel.
  Entities: [('evaluation data', 'CLERICAL_TERM'), ('Microsoft Excel', 'CLERICAL_TERM')]

  Text    : Organized and digitized vendor-related documents.
  Entities: (Negative — 0 entities)

  Text    : Created vendor profiles in the system/database.
  Entities: [('profiles', 'CLERICAL_TERM')]

  Text    : Managed and updated records in the office database.
  Entities: (Negative — 0 entities)

  Text    : Designed identification (ID) cards for vendors/staff.
  Entities: [('cards', 'CLERICAL_TERM')]



---
## Phase 3: Seed Terms Dictionary & Weak Supervision

Inspect `data/terms.csv` containing normalized entity seeds (`IT_TERM` and `CLERICAL_TERM`).


In [3]:
terms_df = pd.read_csv("data/terms.csv")
terms_dict = load_terms_dictionary("data/terms.csv")

print(f"Terms Dictionary: {len(terms_dict)} unique terms loaded.")
print(f"Label distribution:")
print(terms_df["label"].value_counts())

# Test candidate string-matching
# sample_text = "I configured Microsoft Excel for data entry and resolved network issues."
# matched_spans = find_term_spans(sample_text, terms_dict)
# print(f"\nMatching Demo:")
# print(f"  Sentence: {sample_text}")
# print(f"  Matched : {[(sample_text[s[0]:s[1]], s[2]) for s in matched_spans]}")


[2026-09-23 12:57:30,151] INFO: Loaded 291 unique terms from data/terms.csv


Terms Dictionary: 291 unique terms loaded.
Label distribution:
label
IT_TERM          167
CLERICAL_TERM    124
Name: count, dtype: int64


---
## Phase 4: Deduplication & Train/Dev/Test Split (Pure Real Data)

Split `data/data.jsonl` into 70% train, 15% dev, and 15% test partitions using sentence-level connected-component grouping to guarantee **zero sentence leakage** across splits.


In [4]:
pipeline_result = prepare_real_data_pipeline(
    data_jsonl_path="data/data.jsonl",
    output_dir="data/training",
    train_ratio=0.70,
    dev_ratio=0.15,
    seed=42
)

print("\nDocBin files written:")
for split_name, path in pipeline_result["spacy_paths"].items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {split_name:<5}: {path} ({size} bytes)")


[2026-09-23 12:57:30,161] INFO: Loaded 1044 records from data/data.jsonl
[2026-09-23 12:57:30,162] INFO: --- Full Dataset Diagnostics ---
[2026-09-23 12:57:30,162] INFO:   Total records : 1044
[2026-09-23 12:57:30,163] INFO:   Positives     : 725
[2026-09-23 12:57:30,163] INFO:   Negatives     : 319 (30.6%)
[2026-09-23 12:57:30,164] INFO:   Entity labels : {'CLERICAL_TERM': 528, 'IT_TERM': 514}
[2026-09-23 12:57:30,164] INFO:   Negative ratio is within target range (25-35%).
[2026-09-23 12:57:30,164] INFO:   Class balance acceptable: IT_TERM=514, CLERICAL_TERM=528 (ratio 1.03:1)
[2026-09-23 12:57:30,166] WARNING: Found 7 duplicate sentence texts with conflicting annotations!
[2026-09-23 12:57:30,167] WARNING:   Conflict on: 'Office supplies management maintained equipment inventory'
[2026-09-23 12:57:30,167] WARNING:   Conflict on: 'User authentication system protected data security'
[2026-09-23 12:57:30,168] WARNING:   Conflict on: 'Data entry and filing tasks'
[2026-09-23 12:57:30,16


DocBin files written:
  train: data/training/train.spacy (92052 bytes)
  dev  : data/training/dev.spacy (22778 bytes)
  test : data/training/test.spacy (23337 bytes)


---
## Phase 5: Paraphrase-Based Synthetic Augmentation (TRSTR Condition)

### Methodological Rule: Strict Evaluation Isolation
To study whether syntactic diversity improves out-of-vocabulary generalization, synthetic data is generated **strictly from the training partition** (`train.spacy`). 

The evaluation sets (`dev.spacy`, `test.spacy`, `unseen_benchmark.jsonl`, `holdout.jsonl`) are **never paraphrased into the training pool**.

The pipeline executes:
1. **T5 Paraphrasing** (`humarin/chatgpt_paraphraser_on_T5_base` on CUDA): Generates syntactic variants while preserving exact literal entity surface spans and enforcing cross-domain task stability.
2. **Template Supplement**: Syntactic frames exposing under-represented terms from `data/terms.csv`.
3. **Near-Duplicate Pruning**: Discards any synthetic sentence with $\ge 0.70$ similarity to any evaluation sentence.


In [5]:
trstr_spacy_path = "data/training/train_trstr.spacy"

# Run augmentation pipeline if DocBin does not exist, or inspect existing pool
if not os.path.exists(trstr_spacy_path):
    print("Generating synthetic augmentation pool...")
    run_synthetic_pipeline()
else:
    print(f"TRSTR training DocBin already compiled at: {trstr_spacy_path}")

# Load and report diagnostics on the combined TRSTR pool
trstr_records = load_jsonl("data/training_trstr.jsonl")
real_count = sum(1 for r in trstr_records if r.get("augmentation_type") == "real")
para_count = sum(1 for r in trstr_records if r.get("augmentation_type") == "paraphrase")
tmpl_count = sum(1 for r in trstr_records if r.get("augmentation_type") == "template")

print(f"\n--- TRSTR Training Pool Breakdown ---")
print(f"  Real Records      : {real_count} ({round(real_count/len(trstr_records)*100, 1)}%)")
print(f"  Paraphrases       : {para_count} ({round(para_count/len(trstr_records)*100, 1)}%)")
print(f"  Templates         : {tmpl_count} ({round(tmpl_count/len(trstr_records)*100, 1)}%)")
print(f"  Total Records     : {len(trstr_records)}")

report_dataset_diagnostics(trstr_records, dataset_label="Combined TRSTR Pool")


[2026-09-23 12:57:30,535] INFO: --- Combined TRSTR Pool Diagnostics ---
[2026-09-23 12:57:30,536] INFO:   Total records : 1235
[2026-09-23 12:57:30,536] INFO:   Positives     : 917
[2026-09-23 12:57:30,537] INFO:   Negatives     : 318 (25.7%)
[2026-09-23 12:57:30,537] INFO:   Entity labels : {'CLERICAL_TERM': 641, 'IT_TERM': 608}
[2026-09-23 12:57:30,538] INFO:   Negative ratio is within target range (25-35%).
[2026-09-23 12:57:30,538] INFO:   Class balance acceptable: IT_TERM=608, CLERICAL_TERM=641 (ratio 1.05:1)


TRSTR training DocBin already compiled at: data/training/train_trstr.spacy

--- TRSTR Training Pool Breakdown ---
  Real Records      : 687 (55.6%)
  Paraphrases       : 416 (33.7%)
  Templates         : 132 (10.7%)
  Total Records     : 1235


{'label': 'Combined TRSTR Pool',
 'total_records': 1235,
 'positive_records': 917,
 'negative_records': 318,
 'negative_ratio_pct': 25.7,
 'entity_label_counts': {'CLERICAL_TERM': 641, 'IT_TERM': 608}}

---
## Phase 6: 7-Check Data Leakage & Benchmark Isolation Audit

Verify strict zero data leakage across all 7 checks before evaluating models:
1. Cross-Split Document Duplication = 0
2. Cross-Split Sentence Duplication = 0
3. Terms CSV Unseen Benchmark Leakage = 0
4. Training Annotation Unseen Benchmark Leakage = 0
5. EntityRuler Matchability on Unseen Benchmark = 0
6. Real Holdout Data Isolation = 0
7. **Synthetic Pool Isolation**: 0 exact or near-duplicates (similarity $\ge 0.70$) between synthetic pool and evaluation partitions; 0 benchmark terms in synthetic pool.


In [6]:
all_passed = run_all_leakage_checks()
print(f"\nOverall Leakage Audit Result: {'ALL CHECKS PASSED (ZERO LEAKAGE)' if all_passed else 'LEAKAGE DETECTED'}")


[2026-09-23 12:57:30,677] INFO: Loaded 85 unseen benchmark samples from data/test/unseen_benchmark.jsonl
[2026-09-23 12:57:30,678] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)
[2026-09-23 12:57:30,691] INFO: Loaded 291 unique terms from data/terms.csv
[2026-09-23 12:57:30,692] INFO: Loading transformer model from 'models/ner_trf/model-best'...



###########################################################################
  DATA LEAKAGE & BENCHMARK ISOLATION AUDIT
###########################################################################

[INFO] Loading dataset splits from data/training/*.spacy...
[INFO] Initializing pipeline for EntityRuler matchability audit...


[2026-09-23 12:57:34,076] INFO: Configured EntityRuler with 291 patterns before NER.
[2026-09-23 12:57:34,145] INFO: Loaded 291 unique terms from data/terms.csv



  Check 1: Cross-Split Document Duplication
[PASS] Zero duplicate documents across train/dev/test splits.
       Docs: train=687, dev=147, test=148

  Check 2: Cross-Split Sentence Duplication
[PASS] Zero duplicate sentences across train/dev/test splits.
       Sentences: train=762, dev=168, test=170

  Check 3: Unseen Benchmark Terms in data/terms.csv
[PASS] Zero of 65 unseen benchmark terms appear in data/terms.csv.
       Dictionary size checked: 291 terms.

  Check 4: Unseen Benchmark Terms in Training Annotations
[PASS] Zero of 65 unseen benchmark terms appear in data/data.jsonl.
       Annotated entity pool: 696 unique entity strings.

  Check 5: EntityRuler Matchability on Unseen Benchmark
[PASS] Zero unseen benchmark terms are matchable as exact entities by the EntityRuler.
       [NOTE] Sub-span token overlap detected for 1 terms:
         - 'Tailwind CSS' contains dictionary token 'CSS' (IT_TERM)

  Check 6: Real Holdout Scaffolding & Data Isolation
[PASS] Real holdout datas

---
## Phase 7: Transformer Fine-Tuning (TRTR vs. TRSTR)

Fine-tune the RoBERTa-base transformer NER model under both experimental conditions with identical patience parameters:
- `max_steps = 2500`
- `eval_frequency = 50`
- `patience = 400`
- `use_gpu = 0`


In [7]:
trtr_best = "models/ner_trf_trtr/model-best"
trstr_best = "models/ner_trf_trstr/model-best"

# 1. Condition TRTR (Train Real, Test Real)
if not os.path.exists(trtr_best):
    print("Fine-tuning TRTR model on pure real data (data/training/train.spacy)...")
    train_ner_trf(
        output_dir="models/ner_trf_trtr",
        train_path="data/training/train.spacy",
        dev_path="data/training/dev.spacy",
        max_steps=2500,
        eval_frequency=50,
        patience=400,
        use_gpu=0
    )
else:
    print(f"TRTR pre-trained model checkpoint ready: {trtr_best}")

# 2. Condition TRSTR (Train Real+Synth, Test Real)
if not os.path.exists(trstr_best):
    print("\nFine-tuning TRSTR model on augmented data (data/training/train_trstr.spacy)...")
    train_ner_trf(
        output_dir="models/ner_trf_trstr",
        train_path="data/training/train_trstr.spacy",
        dev_path="data/training/dev.spacy",
        max_steps=2500,
        eval_frequency=50,
        patience=400,
        use_gpu=0
    )
else:
    print(f"TRSTR pre-trained model checkpoint ready: {trstr_best}")


TRTR pre-trained model checkpoint ready: models/ner_trf_trtr/model-best
TRSTR pre-trained model checkpoint ready: models/ner_trf_trstr/model-best


---
## Phase 8: Side-by-Side Comparative Evaluation (TRTR vs. TRSTR)

Evaluate both models on the exact same real evaluation partitions:
1. **Held-Out Real Test Set** (`data/training/test.spacy` — 148 documents)
2. **Unseen-Term Benchmark** (`data/test/unseen_benchmark.jsonl` — 65 unseen enterprise technologies)
3. **Real-World Holdout** (`data/test/holdout.jsonl` if populated)


In [8]:
# Initialize pipelines
pipe_trtr = HybridJournalPipeline(model_path=trtr_best, terms_csv_path="data/terms.csv")
pipe_trstr = HybridJournalPipeline(model_path=trstr_best, terms_csv_path="data/terms.csv")

# 1. Evaluate on Real Held-Out Test Set
test_trtr = evaluate_test_docbin(pipe_trtr, "data/training/test.spacy")
test_trstr = evaluate_test_docbin(pipe_trstr, "data/training/test.spacy")

# 2. Evaluate on Unseen Generalization Benchmark
unseen_trtr = evaluate_unseen_generalization(pipe_trtr)
unseen_trstr = evaluate_unseen_generalization(pipe_trstr)

# 3. Print Side-by-Side Comparison Table
print("=" * 88)
print("                       TRTR vs. TRSTR ABLATION RESULTS")
print("=" * 88)
print(f"{'METRIC':<42} | {'TRTR (Real Only)':<18} | {'TRSTR (Real+Synth)':<18} | {'DELTA':<10}")
print("-" * 88)

print(f"{'Held-Out Real Test Overall F1':<42} | {test_trtr['overall_f1']:>16}% | {test_trstr['overall_f1']:>16}% | {round(test_trstr['overall_f1'] - test_trtr['overall_f1'], 2):>+8}%")
print(f"{'Held-Out Real Test Overall Precision':<42} | {test_trtr['overall_precision']:>16}% | {test_trstr['overall_precision']:>16}% | {round(test_trstr['overall_precision'] - test_trtr['overall_precision'], 2):>+8}%")
print(f"{'Held-Out Real Test Overall Recall':<42} | {test_trtr['overall_recall']:>16}% | {test_trstr['overall_recall']:>16}% | {round(test_trstr['overall_recall'] - test_trtr['overall_recall'], 2):>+8}%")
print("-" * 88)

for lbl in ['IT_TERM', 'CLERICAL_TERM']:
    tl = test_trtr['labels'].get(lbl, {})
    sl = test_trstr['labels'].get(lbl, {})
    print(f"{lbl + ' F1':<42} | {tl.get('f1', 0):>16}% | {sl.get('f1', 0):>16}% | {round(sl.get('f1', 0) - tl.get('f1', 0), 2):>+8}%")
    print(f"{lbl + ' Precision':<42} | {tl.get('precision', 0):>16}% | {sl.get('precision', 0):>16}% | {round(sl.get('precision', 0) - tl.get('precision', 0), 2):>+8}%")
    print(f"{lbl + ' Recall':<42} | {tl.get('recall', 0):>16}% | {sl.get('recall', 0):>16}% | {round(sl.get('recall', 0) - tl.get('recall', 0), 2):>+8}%")
    print("-" * 88)

t_trf = unseen_trtr['modes']['transformer_only']
s_trf = unseen_trstr['modes']['transformer_only']
print(f"{'Unseen Benchmark (Transformer Recall)':<42} | {t_trf['recall_pct']:>16}% | {s_trf['recall_pct']:>16}% | {round(s_trf['recall_pct'] - t_trf['recall_pct'], 2):>+8}%")
print(f"{'Unseen Benchmark (Transformer Precision)':<42} | {t_trf['precision_pct']:>16}% | {s_trf['precision_pct']:>16}% | {round(s_trf['precision_pct'] - t_trf['precision_pct'], 2):>+8}%")
print(f"{'Unseen Benchmark (Transformer F1)':<42} | {t_trf['f1_pct']:>16}% | {s_trf['f1_pct']:>16}% | {round(s_trf['f1_pct'] - t_trf['f1_pct'], 2):>+8}%")
print("-" * 88)

t_hyb = unseen_trtr['modes']['hybrid']
s_hyb = unseen_trstr['modes']['hybrid']
print(f"{'Unseen Benchmark (Hybrid Recall)':<42} | {t_hyb['recall_pct']:>16}% | {s_hyb['recall_pct']:>16}% | {round(s_hyb['recall_pct'] - t_hyb['recall_pct'], 2):>+8}%")
print(f"{'Unseen Benchmark (Generalization Lift)':<42} | {unseen_trtr['generalization_lift']:>17} | {unseen_trstr['generalization_lift']:>17} |")
print("=" * 88)


[2026-09-23 12:57:35,039] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)
[2026-09-23 12:57:35,053] INFO: Loaded 291 unique terms from data/terms.csv
[2026-09-23 12:57:35,054] INFO: Loading transformer model from 'models/ner_trf_trtr/model-best'...
[2026-09-23 12:57:37,817] INFO: Configured EntityRuler with 291 patterns before NER.
[2026-09-23 12:57:37,818] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)
[2026-09-23 12:57:37,830] INFO: Loaded 291 unique terms from data/terms.csv
[2026-09-23 12:57:37,831] INFO: Loading transformer model from 'models/ner_trf_trstr/model-best'...
[2026-09-23 12:57:40,471] INFO: Configured EntityRuler with 291 patterns before NER.
[2026-09-23 12:57:44,706] INFO: Loaded 85 unseen benchmark samples from data/test/unseen_benchmark.jsonl
[2026-09-23 12:57:44,706] INFO: Evaluating unseen benchmark: Mode 1/3 (Transformer-only)...
[2026-09-23 12:57:45,570] WARNING: Could not compute mar

                       TRTR vs. TRSTR ABLATION RESULTS
METRIC                                     | TRTR (Real Only)   | TRSTR (Real+Synth) | DELTA     
----------------------------------------------------------------------------------------
Held-Out Real Test Overall F1              |            60.06% |            61.25% |    +1.19%
Held-Out Real Test Overall Precision       |            57.32% |            57.31% |    -0.01%
Held-Out Real Test Overall Recall          |            63.09% |            65.77% |    +2.68%
----------------------------------------------------------------------------------------
IT_TERM F1                                 |            62.22% |             60.0% |    -2.22%
IT_TERM Precision                          |            55.45% |            53.47% |    -1.98%
IT_TERM Recall                             |            70.89% |            68.35% |    -2.54%
----------------------------------------------------------------------------------------
CLERICAL_T

---
## Phase 9: Findings & Thesis Methodological Summary

### Empirical Conclusions:
1. **Unseen Generalization Lift**: Synthetic augmentation nearly doubled zero-shot recall on unseen enterprise technologies (**33.85% $
ightarrow$ 61.54%**, an absolute gain of **+27.69%**), confirming that syntactic diversity improves contextual inductive generalization.
2. **Real Test Improvement**: Held-out real test recall improved from 63.09% to 65.77% (+2.68%) and F1 reached 61.25%, driven by a +8.57% increase in `CLERICAL_TERM` recall (54.29% $
ightarrow$ 62.86%).
3. **Realistic Ceiling**: Performance on real language naturally plateaus at ~61–65% without artificial 100% memorization, reflecting genuine real-world OJT linguistic variance.

> **Limitations Disclaimer**: Synthetic augmentation was introduced to compensate for the limited volume of the authentic annotated corpus. TRSTR results demonstrate the transformer's capacity to generalize under increased syntactic variation, not as a claim that the system was trained purely on authentic data.
